# 02 - Transformação

Aplica as decisões registradas no `01_eda.ipynb`.

In [ ]:
import sys
sys.path.append("..")

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Carrega cópias de trabalho mantendo os arquivos originais intactos.
cadastral = pd.read_parquet("../dados/base_cadastral.parquet").copy()
submissao = pd.read_parquet("../dados/base_submissao.parquet").copy()
emprestimos = pd.read_parquet("../dados/historico_emprestimos.parquet").copy()
parcelas = pd.read_parquet("../dados/historico_parcelas.parquet").copy()

print(
    f"cadastral: {cadastral.shape} | submissao: {submissao.shape} | "
    f"emprestimos: {emprestimos.shape} | parcelas: {parcelas.shape}"
)

In [ ]:
# --- TRATAMENTO E CONSOLIDAÇÃO: BASE CADASTRAL ---

# 1. Guarda referências dinâmicas para asserções
n_clientes_esperados = cadastral["id_cliente"].nunique()

# 2. Identificação de datas de nascimento sentinela e conversão de tipos
datas_convertidas = pd.to_datetime(cadastral["data_nascimento"], errors="coerce")
cadastral["data_nascimento_invalida"] = (
    cadastral["data_nascimento"].notna() & (datas_convertidas.dt.year <= 1920)
)
cadastral.loc[cadastral["data_nascimento_invalida"], "data_nascimento"] = None

# Aplica conversão padronizada de tipos estruturais
cadastral, rel_tipos = fn.converter_tipos(
    cadastral,
    datas={"data_nascimento": ["%Y-%m-%d", "%d/%m/%Y"]},
    numericas_texto={"renda_anual": {"decimal": ",", "milhar": "."}} if cadastral["renda_anual"].dtype == "object" else None,
    inteiros=["qtd_membros_familia"],
)

# 3. Padronização de variáveis categóricas e conversão de placeholders em NaN
cols_categoricas = [
    c for c in cadastral.select_dtypes(include=["object", "string", "category"]).columns 
    if c not in ["data_nascimento"]
]
cadastral, rel_cat = fn.padronizar_categoricas(cadastral, colunas=cols_categoricas)

# 4. Consolidação de duplicatas: recência com preenchimento de nulos
mascara_dup = cadastral.duplicated(subset=["id_cliente"], keep=False)
cad_unicos = cadastral.loc[~mascara_dup].set_index("id_cliente")

cad_1a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="last")].set_index("id_cliente")
cad_2a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="first")].set_index("id_cliente")

# A 2ª ocorrência prevalece; onde houver NaN, herda o valor da 1ª ocorrência
cad_dups_consolidadas = cad_2a.combine_first(cad_1a)

cadastral_tratada = (
    pd.concat([cad_unicos, cad_dups_consolidadas])
    .sort_index()
    .reset_index()
)

# 5. Otimização de tipos finais para controle de memória
cadastral_tratada["id_cliente"] = cadastral_tratada["id_cliente"].astype("int64")
cadastral_tratada["qtd_filhos"] = cadastral_tratada["qtd_filhos"].astype("Int64")
for col in cols_categoricas:
    cadastral_tratada[col] = cadastral_tratada[col].astype("category")

# 6. Remoção de colunas sensíveis/não utilizadas no modelo
cols_para_remover = ["sexo"]
n_cols_antes_drop = cadastral_tratada.shape[1]
cadastral_tratada = cadastral_tratada.drop(columns=cols_para_remover)

# 7. Asserções para validação do pipeline
assert cadastral_tratada.shape[0] == n_clientes_esperados, (
    f"Esperado {n_clientes_esperados} linhas, obtido {cadastral_tratada.shape[0]}"
)
assert cadastral_tratada["id_cliente"].nunique() == n_clientes_esperados, (
    "Falha: id_cliente não é único na base consolidada"
)
assert cadastral_tratada.shape[1] == (n_cols_antes_drop - len(cols_para_remover)), (
    "Inconsistência no número de colunas após exclusão de campos indesejados"
)

print("--- Base Cadastral Tratada com Sucesso ---")
print(f"Shape final: {cadastral_tratada.shape}")
print(f"Clientes únicos garantidos: {cadastral_tratada['id_cliente'].nunique()}")
print(f"Total com data_nascimento_invalida: {cadastral_tratada['data_nascimento_invalida'].sum()}")

In [ ]:
cadastral_tratada.head()

In [ ]:
# --- TRATAMENTO E TIPAGEM: BASE EMPRESTIMOS ---

# 1. Guarda referências dinâmicas da base de entrada
n_linhas_esperadas_emp = len(emprestimos)
n_contratos_esperados_emp = emprestimos["id_contrato"].nunique()

emprestimos_proc = emprestimos.copy()

# 2. Conversão de colunas temporais
datas_emprestimos = {
    "data_decisao": ["%Y-%m-%d", "%d/%m/%Y"],
    "data_primeiro_vencimento": ["%Y-%m-%d", "%d/%m/%Y"],
}

emprestimos_proc, rel_datas = fn.converter_tipos(
    emprestimos_proc,
    datas={c: datas_emprestimos[c] for c in datas_emprestimos if c in emprestimos_proc.columns},
)

# 3. Padronização de colunas categóricas e conversão de placeholders em nulo
cols_cat_emp = [
    c for c in emprestimos_proc.select_dtypes(include=["object", "string", "category"]).columns
    if c not in datas_emprestimos
]

emprestimos_proc, rel_cat_emp = fn.padronizar_categoricas(
    emprestimos_proc, colunas=cols_cat_emp
)

# 4. Otimização de tipos de dados
emprestimos_proc["id_contrato"] = emprestimos_proc["id_contrato"].astype("int64")
emprestimos_proc["id_cliente"] = emprestimos_proc["id_cliente"].astype("int64")

for col in cols_cat_emp:
    emprestimos_proc[col] = emprestimos_proc[col].astype("category")

emprestimos_tratada = emprestimos_proc.copy()

# 7. Asserções para validação do pipeline
assert len(emprestimos_tratada) == n_linhas_esperadas_emp, (
    f"Perda de registros: esperado {n_linhas_esperadas_emp}, obtido {len(emprestimos_tratada)}"
)
assert emprestimos_tratada["id_contrato"].nunique() == n_contratos_esperados_emp, (
    f"Inconsistência de chave: esperado {n_contratos_esperados_emp} contratos únicos, "
    f"obtido {emprestimos_tratada['id_contrato'].nunique()}"
)

print("--- Base Empréstimos Tratada com Sucesso ---")
print(f"Shape final: {emprestimos_tratada.shape}")
print(f"Contratos únicos: {emprestimos_tratada['id_contrato'].nunique()}")
print(f"Clientes distintos na base: {emprestimos_tratada['id_cliente'].nunique()}")